Copyright 2026 DataRobot, Inc. and its affiliates.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

In [ ]:
from jointfm_client import bootstrap_notebook

bootstrap_notebook(add_src_root=True)

# Service Health
Fetch the typed `/healthz` payload from the configured JointFM deployment and display every field. Use this notebook to diagnose health-endpoint issues: version pins, advertised modes, `decoding_strategy` (parallel vs autoregressive horizon decoding), sample-count budgets, and the optional `data_generation` capability block.

`JOINTFM_DEPLOYMENT_IDS` configures a round-robin pool of hosted deployments. Each endpoint's health payload describes only that endpoint, so the client probes every configured peer and aggregates locally: `health()` returns consensus metadata whose `max_sample_count` is the **minimum** reachable cap, which is the sample-batch size used to split oversized sample requests. `cache=True` stores this probe so the topology section below reuses it instead of issuing a second round of requests.

In [ ]:
from dataclasses import asdict
from pprint import pprint

from jointfm_client import JointFMClient

client = JointFMClient.from_env()
health = client.health(cache=True, refresh=True)
pprint(asdict(health), sort_dicts=False, width=100)

## Deployment Topology
`health_instances()` returns the per-endpoint results behind that consensus: one entry per configured deployment ID, including peers skipped as unreachable or contract-incompatible. Its `max_sample_count` is the **sum** of reachable caps, the overall parallel capacity of the pool, and `topology_label` groups those caps as `<count>x<cap>`. Reachable peers must agree on `model_version` and `checkpoint_version`; a mismatch fails the probe rather than silently mixing models across requests.

In [ ]:
settings = client.settings
instances = client.health_instances(cache=True)
# A pool repeats its primary endpoint in `instances`; the seed covers single-endpoint configs.
predict_url_by_id: dict[str | None, str] = {
    settings.deployment_id: settings.predict_url
}
for instance in settings.instances:
    predict_url_by_id[instance.deployment_id] = instance.predict_url

print(f"selector:             {settings.deployment_selector}")
print(f"configured endpoints: {len(instances.instances)}")
print(f"topology:             {instances.topology_label}")
print(f"parallel capacity:    {instances.max_sample_count} (sum of reachable caps)")
print(f"sample-batch cap:     {health.max_sample_count} (minimum reachable cap)")

for instance in instances.instances:
    print()
    if instance.metadata is None:
        print(f"{instance.deployment_id}: unavailable")
        print(f"  error:      {instance.error}")
        continue
    print(f"{instance.deployment_id}: available")
    print(f"  url:        {predict_url_by_id[instance.deployment_id]}")
    print(f"  device:     {instance.metadata.device}")
    print(f"  image:      {instance.metadata.image_version}")
    print(f"  checkpoint: {instance.metadata.checkpoint_version}")
    print(f"  samples:    {instance.metadata.max_sample_count}")